In [ ]:
import os
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, AIMessage

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# Placeholder for LLM and embeddings
def setup_models():
    llm = ChatOllama(model="deepseek-r1:7b", temperature=0.9)
    embeddings = OllamaEmbeddings(model="deepseek-r1:7b")
    return llm, embeddings

# Initialize LLM and embeddings
llm, embeddings = setup_models()

# Document loading configuration
FILE_PATH = "a.pdf"  # Replace with the path to your document
EXPORT_TYPE = ExportType.DOC_CHUNKS  # Chunked export mode

# Load documents using DoclingLoader
loader = DoclingLoader(
    file_path=FILE_PATH,
    export_type=EXPORT_TYPE,
)
docs = loader.load()

# Split documents (if needed)
splits = docs

# Initialize Chroma for vector storage
vectorstore = Chroma(
    collection_name="demo_collection",
    embedding_function=embeddings,  # Use the embedding model here
    persist_directory="./chroma_demo_db",
)

# Add documents to Chroma
vectorstore.add_documents(splits)

# Setup retriever from Chroma
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

# Example tool configuration
def create_retriever_tool():
    return Tool(
        name="Document Retriever",
        description="Retrieve relevant documents based on a query.",
        func=lambda query: retriever.retrieve(query),
    )

# Create tools
tools = [create_retriever_tool()]

# Main processing function
def process_query(query):
    human_message = HumanMessage(content=query)
    # Placeholder response logic
    response = {"content": "Response placeholder.", "retrieved_docs": []}
    return response

# Placeholder for query execution
if __name__ == "__main__":
    query = "hi"
    response = process_query(query)
    print("Response:", response)


In [3]:
from docling.document_converter import DocumentConverter
import os
import gc
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

source = "b.pdf"  # Document path or URL
output_dir = "output_images"  # Directory to save extracted images
os.makedirs(output_dir, exist_ok=True)  # Create directory if it doesn't exist

# Initialize the document converter
converter = DocumentConverter()
result = converter.convert(source)

# Save the result to a text file with utf-8 encoding
with open("output.txt", "w", encoding="utf-8") as file:
    file.write(result.document.export_to_markdown())

# Save extracted images
picture_counter = 0
for element, _level in result.document.iterate_items():
    if hasattr(element, "get_image"):  # Check if the element has an image
        picture_counter += 1
        image_filename = os.path.join(output_dir, f"image_{picture_counter}.png")
        image = element.get_image(result.document)  # Get the image
        if image is not None:
            with open(image_filename, "wb") as img_file:
                image.save(img_file, format="PNG")

# Free up GPU memory
torch.cuda.empty_cache()

print(f"Result saved to output.txt and {picture_counter} images saved to {output_dir}.")


Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAt

Result saved to output.txt and 18 images saved to output_images.


In [18]:
def count_image_tags(file_path):
    tag = "<!-- image -->"
    count = 0

    with open(file_path, 'r') as file:
        for line in file:
            count += line.count(tag)
    
    return count

if __name__ == "__main__":
    file_path = 'output.txt'
    tag_count = count_image_tags(file_path)
    print(f"The number of '<!-- image -->' tags in {file_path} is: {tag_count}")

The number of '<!-- image -->' tags in output.txt is: 149


: 

In [8]:
from docling.document_converter import DocumentConverter
import os
import gc
import torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

source = "a.pdf"  # document per local path or URL
converter = DocumentConverter()
result = converter.convert(source)

# Save the result to a text file with utf-8 encoding
with open("output.txt", "w", encoding="utf-8") as file:
    file.write(result.document.export_to_markdown())

# Free up GPU memory
torch.cuda.empty_cache()

print("Result saved to output.txt")

Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAttention: The specified module could not be found.
Could not load the custom kernel for multi-scale deformable attention: DLL load failed while importing MultiScaleDeformableAt

Result saved to output.txt


In [4]:
import os
import json
from langchain.docstore.document import Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings, ChatOllama
from collections import OrderedDict
from typing import List, Optional

# Assuming an embedding model instance is available
embed_model = OllamaEmbeddings(model="deepseek-r1:7b")

# Define file paths
DOCUMENTS_DIR = "documents"
METADATA_DIR = "metadata"
OUTPUT_FILE = "output_chunks.txt"

# Ensure output file is empty before writing
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("")
class DocPOIDirectoryLoader:
    """Loads and processes text documents from a directory along with metadata."""

    def __init__(self, directory_path: str, metadata_path: Optional[str] = None) -> None:
        self.directory_path = directory_path
        self.metadata_path = metadata_path or directory_path

    def load(self) -> List[Document]:
        """Loads all text documents in the directory and returns a list of Document objects."""
        documents = []
        for filename in os.listdir(self.directory_path):
            if filename.endswith('.txt'):
                file_path = os.path.join(self.directory_path, filename)
                metadata_file = os.path.join(self.metadata_path, f"{os.path.splitext(filename)[0]}.json")

                metadata = self.load_metadata(metadata_file)
                metadata['document_id'] = metadata.get('document_id', os.path.splitext(filename)[0])

                documents.extend(self.load_text(file_path, metadata))
        return documents

    def load_metadata(self, metadata_file: str) -> dict:
        """Loads metadata from a JSON file if available."""
        if os.path.exists(metadata_file):
            with open(metadata_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        return {}

    def sanitize_metadata(self, metadata: dict) -> dict:
        """Ensures all metadata values are of type str, int, float, or bool."""
        sanitized_metadata = {}
        for key, value in metadata.items():
            if isinstance(value, list):
                sanitized_metadata[key] = ", ".join(map(str, value))  # Convert list to comma-separated string
            elif isinstance(value, (str, int, float, bool)):
                sanitized_metadata[key] = value  # Keep valid types
            else:
                sanitized_metadata[key] = str(value)  # Convert other types to string
        return sanitized_metadata

    def load_text(self, file_path: str, metadata: dict) -> List[Document]:
        """Reads a text file, chunks it, and creates Document objects."""
        with open(file_path, 'r', encoding='utf-8') as f:
            text_content = f.read()

        embedding = embed_model
        text_splitter = SemanticChunker(embedding, breakpoint_threshold_type="gradient", min_chunk_size=1000)
        chunks = text_splitter.create_documents([text_content])

        sanitized_metadata = self.sanitize_metadata(metadata)  # Sanitize metadata before passing

        return [
            Document(
                page_content=chunk.page_content,
                metadata=OrderedDict(sanitized_metadata, page_number=page_number + 1, source=file_path)
            ) for page_number, chunk in enumerate(chunks)
        ]

# Instantiate loader
loader = DocPOIDirectoryLoader(DOCUMENTS_DIR, METADATA_DIR)

# Load and process documents
documents = loader.load()

# Save chunks to a text file
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(f"--- Start of Chunk ---\n")
        f.write(f"Document ID: {doc.metadata['document_id']}\n")
        f.write(f"Page Number: {doc.metadata['page_number']}\n")
        f.write(f"Source: {doc.metadata['source']}\n")
        f.write(f"Content:\n{doc.page_content}\n")
        f.write(f"--- End of Chunk ---\n\n")

print(f"Chunks saved to {OUTPUT_FILE}")


Chunks saved to output_chunks.txt


In [1]:
from components.record_manager import (
    initialize_vectorstore, 
    add_folder_to_vectorstore, 
    add_file_to_vectorstore, 
    reset_vectorstore
) 
import os


add_folder_to_vectorstore("documents", "metadata")

Indexing 3 documents from documents...
Error indexing documents: {} (status code: 500)


ResponseError: {} (status code: 500)